# Bird Classification - Systematic Experiments

## Experiment Matrix

**Models**: Swin Tiny, ViT-Base  
**Per Model Experiments**:
1. Baseline (freeze all, train head only)
2. Full Fine-tuning (unfreeze all layers)
3. Progressive Unfreezing (unfreeze last 2 blocks)
4. Best + TTA (apply TTA to best performing config)

**Cross-Model**:
5. Ensemble (best Swin + best ViT)


In [ ]:
# GPU Verification
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install timm -q

In [ ]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json
from datetime import datetime

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from PIL import Image

import timm
from sklearn.model_selection import StratifiedKFold
from tqdm.auto import tqdm

import warnings
warnings.filterwarnings('ignore')

def set_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)

set_seed(42)

In [ ]:
# Paths
BASE_PATH = Path("/content/drive/MyDrive/bird_classification/bdma-07-competition-2026/BDMA7_project_files")
TRAIN_IMG_DIR = BASE_PATH / "train_images"
VAL_IMG_DIR = BASE_PATH / "val_images"
TEST_IMG_DIR = BASE_PATH / "test_images/mistery_cat"
OUTPUT_DIR = BASE_PATH / "experiment_results"
OUTPUT_DIR.mkdir(exist_ok=True)

print(f"Data directory exists: {BASE_PATH.exists()}")

## Configuration

In [ ]:
class ExperimentConfig:
    """Base configuration for all experiments"""
    # Models
    models = ['swin_tiny_patch4_window7_224', 'vit_base_patch16_224']

    # Data
    img_size = 224
    batch_size = 32
    num_classes = 20
    n_folds = 3

    # Training
    freeze_epochs = 15
    finetune_epochs = 10
    lr_head = 1e-3
    lr_backbone_swin = 1e-5
    lr_backbone_vit = 1e-4
    weight_decay = 1e-4
    label_smoothing = 0.1
    patience = 10

    # Progressive unfreezing
    unfreeze_last_n_blocks = 2

    # TTA
    tta_n_augments = 5

config = ExperimentConfig()
print("Config loaded")

## Load Data

In [ ]:
# Load and combine train/val for K-fold
train_df = pd.read_csv(BASE_PATH / "train_metadata.csv")
val_df = pd.read_csv(BASE_PATH / "val_metadata.csv")

# Check actual column names
print(f"Train columns: {train_df.columns.tolist()}")
print(f"Val columns: {val_df.columns.tolist()}")

# Detect column names
img_col = 'image_name' if 'image_name' in train_df.columns else train_df.columns[0]
class_col = 'class' if 'class' in train_df.columns else train_df.columns[1]

train_df['full_path'] = train_df[img_col].apply(lambda x: str(TRAIN_IMG_DIR / x))
val_df['full_path'] = val_df[img_col].apply(lambda x: str(VAL_IMG_DIR / x))

combined_df = pd.concat([train_df, val_df], ignore_index=True)

# Class mapping
class_mapping = {name: idx for idx, name in enumerate(sorted(combined_df[class_col].unique()))}
combined_df['label'] = combined_df[class_col].map(class_mapping).astype('int64')  # FIXED: Explicit int64

print(f"Total samples: {len(combined_df)}")
print(f"Classes: {len(class_mapping)}")
print(f"Label dtype: {combined_df['label'].dtype}")

## Data Pipeline

In [ ]:
# Transforms
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = T.Compose([
    T.Resize((config.img_size, config.img_size)),
    T.RandomHorizontalFlip(0.5),
    T.RandomRotation(10),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    T.RandomErasing(p=0.2, scale=(0.02, 0.2), value='random')
])

val_transform = T.Compose([
    T.Resize((config.img_size, config.img_size)),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])

# TTA transforms
tta_transforms = [
    val_transform,  # Original
    T.Compose([T.Resize((config.img_size, config.img_size)), T.RandomHorizontalFlip(1.0), T.ToTensor(), T.Normalize(IMAGENET_MEAN, IMAGENET_STD)]),
    T.Compose([T.Resize((config.img_size, config.img_size)), T.RandomRotation(10), T.ToTensor(), T.Normalize(IMAGENET_MEAN, IMAGENET_STD)]),
    T.Compose([T.Resize((config.img_size, config.img_size)), T.RandomRotation((-10, 10)), T.ToTensor(), T.Normalize(IMAGENET_MEAN, IMAGENET_STD)]),
    T.Compose([T.Resize((config.img_size, config.img_size)), T.ColorJitter(brightness=0.1), T.ToTensor(), T.Normalize(IMAGENET_MEAN, IMAGENET_STD)])
]

class BirdDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = self.df.loc[idx, 'full_path']
        label = int(self.df.loc[idx, 'label'])  # FIXED: Explicit int conversion
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, label

print("Dataset class defined")

## Model & Training Functions

In [ ]:
def create_model(model_name, num_classes):
    """Create model with custom head"""
    # Create model with the correct number of classes directly
    # timm will handle creating the proper classifier
    model = timm.create_model(model_name, pretrained=True, num_classes=num_classes)

    # For Swin and ViT, timm already creates a proper Linear classifier
    # We just need to add dropout before it
    n_features = model.num_features

    # Check which attribute the model uses and add dropout
    if hasattr(model, 'head') and isinstance(model.head, nn.Linear):
        # Replace with dropout + linear
        model.head = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(n_features, num_classes)
        )
    elif hasattr(model, 'fc') and isinstance(model.fc, nn.Linear):
        model.fc = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(n_features, num_classes)
        )
    elif hasattr(model, 'classifier') and isinstance(model.classifier, nn.Linear):
        model.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(n_features, num_classes)
        )

    return model

def freeze_backbone(model):
    """Freeze all layers except head"""
    for param in model.parameters():
        param.requires_grad = False

    # Unfreeze the classifier (different models use different names)
    if hasattr(model, 'head'):
        for param in model.head.parameters():
            param.requires_grad = True
    elif hasattr(model, 'fc'):
        for param in model.fc.parameters():
            param.requires_grad = True
    elif hasattr(model, 'classifier'):
        for param in model.classifier.parameters():
            param.requires_grad = True

def unfreeze_all(model):
    """Unfreeze all layers"""
    for param in model.parameters():
        param.requires_grad = True

def unfreeze_last_n_blocks(model, model_name, n_blocks=2):
    """Unfreeze only last N blocks + head"""
    # First freeze all
    for param in model.parameters():
        param.requires_grad = False

    # Unfreeze head (different models use different names)
    if hasattr(model, 'head'):
        for param in model.head.parameters():
            param.requires_grad = True
    elif hasattr(model, 'fc'):
        for param in model.fc.parameters():
            param.requires_grad = True
    elif hasattr(model, 'classifier'):
        for param in model.classifier.parameters():
            param.requires_grad = True

    # Unfreeze last N blocks
    if 'swin' in model_name:
        # Swin has 'layers' attribute
        if hasattr(model, 'layers'):
            layers = list(model.layers)
            for layer in layers[-n_blocks:]:
                for param in layer.parameters():
                    param.requires_grad = True
    elif 'vit' in model_name:
        # ViT has 'blocks' attribute
        if hasattr(model, 'blocks'):
            blocks = list(model.blocks)
            for block in blocks[-n_blocks:]:
                for param in block.parameters():
                    param.requires_grad = True

print("Model functions defined")

In [ ]:
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, correct, total = 0, 0, 0

    for images, labels in tqdm(loader, leave=False):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    return total_loss / len(loader), 100. * correct / total

@torch.no_grad()
def validate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0, 0, 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)

        total_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    return total_loss / len(loader), 100. * correct / total

print("Training functions defined")

## Experiment Runner

In [ ]:
def run_experiment(model_name, experiment_type, fold_idx, train_loader, val_loader, device):
    """
    Run a single experiment

    experiment_type: 'baseline', 'full_finetune', 'progressive_unfreeze'
    """
    print(f"\n{'='*60}")
    print(f"Experiment: {experiment_type} | Model: {model_name} | Fold: {fold_idx}")
    print(f"{'='*60}")

    # Create model
    model = create_model(model_name, config.num_classes).to(device)

    # Get learning rates
    lr_backbone = config.lr_backbone_swin if 'swin' in model_name else config.lr_backbone_vit

    # Setup based on experiment type
    if experiment_type == 'baseline':
        # Train head only
        freeze_backbone(model)
        # Get classifier parameters (different models use different names)
        if hasattr(model, 'head'):
            head_params = model.head.parameters()
        elif hasattr(model, 'fc'):
            head_params = model.fc.parameters()
        elif hasattr(model, 'classifier'):
            head_params = model.classifier.parameters()
        else:
            head_params = model.parameters()

        optimizer = torch.optim.AdamW(head_params, lr=config.lr_head, weight_decay=config.weight_decay)
        total_epochs = config.freeze_epochs

    elif experiment_type == 'full_finetune':
        # Stage 1: Train head
        freeze_backbone(model)
        # Get classifier parameters (different models use different names)
        if hasattr(model, 'head'):
            head_params = model.head.parameters()
        elif hasattr(model, 'fc'):
            head_params = model.fc.parameters()
        elif hasattr(model, 'classifier'):
            head_params = model.classifier.parameters()
        else:
            head_params = model.parameters()

        optimizer = torch.optim.AdamW(head_params, lr=config.lr_head, weight_decay=config.weight_decay)
        # Stage 2 will unfreeze all
        total_epochs = config.freeze_epochs + config.finetune_epochs

    elif experiment_type == 'progressive_unfreeze':
        # Stage 1: Train head
        freeze_backbone(model)
        # Get classifier parameters (different models use different names)
        if hasattr(model, 'head'):
            head_params = model.head.parameters()
        elif hasattr(model, 'fc'):
            head_params = model.fc.parameters()
        elif hasattr(model, 'classifier'):
            head_params = model.classifier.parameters()
        else:
            head_params = model.parameters()

        optimizer = torch.optim.AdamW(head_params, lr=config.lr_head, weight_decay=config.weight_decay)
        # Stage 2 will unfreeze last N blocks
        total_epochs = config.freeze_epochs + config.finetune_epochs

    criterion = nn.CrossEntropyLoss()  # Removed label_smoothing - can add back if needed
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, total_epochs)

    # Training loop
    best_val_acc = 0
    best_model_state = None
    patience_counter = 0
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

    for epoch in range(total_epochs):
        # Stage transition
        if epoch == config.freeze_epochs and experiment_type != 'baseline':
            print(f"\n>>> Transitioning to Stage 2 at epoch {epoch}")

            if experiment_type == 'full_finetune':
                unfreeze_all(model)
            elif experiment_type == 'progressive_unfreeze':
                unfreeze_last_n_blocks(model, model_name, config.unfreeze_last_n_blocks)

            # New optimizer with different param groups
            trainable_params = [p for p in model.parameters() if p.requires_grad]
            optimizer = torch.optim.AdamW(trainable_params, lr=lr_backbone, weight_decay=config.weight_decay)
            scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, total_epochs - epoch)

        # Train & validate
        train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_acc = validate(model, val_loader, criterion, device)
        scheduler.step()

        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)

        print(f"Epoch {epoch+1}/{total_epochs} | Train: {train_acc:.2f}% | Val: {val_acc:.2f}%")

        # Early stopping & best model tracking
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_model_state = model.state_dict().copy()
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= config.patience:
                print(f"Early stopping at epoch {epoch+1}")
                break

    # Load best model
    model.load_state_dict(best_model_state)

    print(f" Best Val Acc: {best_val_acc:.2f}%")

    return {
        'model': model,
        'best_val_acc': best_val_acc,
        'history': history
    }

print("Experiment runner defined")

## TTA & Ensemble Functions

In [ ]:
@torch.no_grad()
def predict_with_tta(model, image_path, transforms_list, device):
    """Apply TTA and average predictions"""
    model.eval()
    image = Image.open(image_path).convert('RGB')

    all_preds = []
    for transform in transforms_list:
        img_tensor = transform(image).unsqueeze(0).to(device)
        output = model(img_tensor)
        probs = torch.softmax(output, dim=1)
        all_preds.append(probs.cpu())

    # Average predictions
    avg_pred = torch.stack(all_preds).mean(0)
    return avg_pred.squeeze(0)

@torch.no_grad()
def ensemble_predict(models, image_path, transform, device, weights=None):
    """Ensemble predictions from multiple models"""
    image = Image.open(image_path).convert('RGB')
    img_tensor = transform(image).unsqueeze(0).to(device)

    all_preds = []
    for model in models:
        model.eval()
        output = model(img_tensor)
        probs = torch.softmax(output, dim=1)
        all_preds.append(probs.cpu())

    # Weighted average if weights provided
    if weights is not None:
        weights_tensor = torch.tensor(weights).view(-1, 1, 1)
        weighted_preds = torch.stack(all_preds) * weights_tensor
        avg_pred = weighted_preds.sum(0)
    else:
        avg_pred = torch.stack(all_preds).mean(0)

    return avg_pred.squeeze(0)

print("TTA/Ensemble functions defined")

## Per-Class Metrics Evaluation

Functions to extract detailed per-class performance metrics

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import pandas as pd
import numpy as np

def evaluate_per_class_metrics(model, dataloader, class_names, device, model_name="Model"):
    """
    Evaluate per-class metrics for a model
    
    Args:
        model: trained model
        dataloader: validation/test dataloader
        class_names: list of class names
        device: torch device
        model_name: name for display
    
    Returns:
        dict with per-class metrics
    """
    model.eval()
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for images, labels in tqdm(dataloader, desc=f"Evaluating {model_name}"):
            images = images.to(device)
            labels = labels.to(device)
            
            outputs = model(images)
            if hasattr(outputs, 'logits'):
                outputs = outputs.logits
            
            _, preds = torch.max(outputs, 1)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    # Calculate overall accuracy
    accuracy = np.mean(np.array(all_preds) == np.array(all_labels))
    
    # Classification report
    report = classification_report(all_labels, all_preds, 
                                   target_names=class_names, 
                                   output_dict=True, 
                                   zero_division=0)
    
    # Confusion matrix
    cm = confusion_matrix(all_labels, all_preds)
    
    # Per-class accuracies
    per_class_acc = {}
    for i, class_name in enumerate(class_names):
        class_mask = np.array(all_labels) == i
        if class_mask.sum() > 0:
            class_correct = np.sum((np.array(all_preds)[class_mask] == i))
            per_class_acc[class_name] = class_correct / class_mask.sum() * 100
        else:
            per_class_acc[class_name] = 0.0
    
    return {
        'accuracy': accuracy,
        'per_class_accuracy': per_class_acc,
        'classification_report': report,
        'confusion_matrix': cm,
        'predictions': all_preds,
        'labels': all_labels
    }

def print_per_class_results(metrics, class_names, model_name="Model"):
    """
    Print formatted per-class results
    """
    print("="*80)
    print(f"{model_name} - PER-CLASS PERFORMANCE")
    print("="*80)
    print(f"\nOverall Accuracy: {metrics['accuracy']*100:.2f}%\n")
    
    print(f"{'Class':<30} {'Accuracy':<12} {'Precision':<12} {'Recall':<12} {'F1-Score':<12}")
    print("-"*80)
    
    report = metrics['classification_report']
    for class_name in sorted(class_names):
        if class_name in report:
            acc = metrics['per_class_accuracy'][class_name]
            prec = report[class_name]['precision'] * 100
            rec = report[class_name]['recall'] * 100
            f1 = report[class_name]['f1-score'] * 100
            print(f"{class_name:<30} {acc:>6.1f}%      {prec:>6.1f}%      {rec:>6.1f}%      {f1:>6.1f}%")
    
    print("\n" + "="*80)

def save_per_class_metrics_csv(metrics_dict, output_path):
    """
    Save per-class metrics to CSV
    
    Args:
        metrics_dict: dict with model_name -> metrics mapping
        output_path: path to save CSV
    """
    rows = []
    
    # Get all class names from first model
    first_model = list(metrics_dict.keys())[0]
    class_names = sorted(metrics_dict[first_model]['per_class_accuracy'].keys())
    
    for class_name in class_names:
        row = {'Class': class_name}
        for model_name, metrics in metrics_dict.items():
            acc = metrics['per_class_accuracy'].get(class_name, 0)
            row[f"{model_name}_Accuracy"] = f"{acc:.1f}"
            
            report = metrics['classification_report']
            if class_name in report:
                row[f"{model_name}_Precision"] = f"{report[class_name]['precision']*100:.1f}"
                row[f"{model_name}_Recall"] = f"{report[class_name]['recall']*100:.1f}"
                row[f"{model_name}_F1"] = f"{report[class_name]['f1-score']*100:.1f}"
        rows.append(row)
    
    df = pd.DataFrame(rows)
    df.to_csv(output_path, index=False)
    print(f"\nPer-class metrics saved to: {output_path}")
    return df


## Run All Experiments

In [ ]:
# Storage for all results
all_results = {}

# Define experiments (excluding TTA for now - will be applied later)
experiments = ['baseline', 'full_finetune', 'progressive_unfreeze']

# K-Fold split
skf = StratifiedKFold(n_splits=config.n_folds, shuffle=True, random_state=42)

for model_name in config.models:
    model_type = 'Swin' if 'swin' in model_name else 'ViT'
    all_results[model_name] = {}

    print(f"\n\n{'#'*70}")
    print(f"MODEL: {model_type}")
    print(f"{'#'*70}")

    for exp_name in experiments:
        all_results[model_name][exp_name] = {'folds': []}

        for fold_idx, (train_idx, val_idx) in enumerate(skf.split(combined_df, combined_df['label'])):
            train_fold_df = combined_df.iloc[train_idx]
            val_fold_df = combined_df.iloc[val_idx]

            train_dataset = BirdDataset(train_fold_df, train_transform)
            val_dataset = BirdDataset(val_fold_df, val_transform)

            train_loader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True, num_workers=2)
            val_loader = DataLoader(val_dataset, batch_size=config.batch_size, shuffle=False, num_workers=2)

            # Run experiment
            result = run_experiment(model_name, exp_name, fold_idx, train_loader, val_loader, device)
            all_results[model_name][exp_name]['folds'].append(result)

        # Calculate mean performance
        fold_accs = [f['best_val_acc'] for f in all_results[model_name][exp_name]['folds']]
        mean_acc = np.mean(fold_accs)
        std_acc = np.std(fold_accs)
        all_results[model_name][exp_name]['mean_acc'] = mean_acc
        all_results[model_name][exp_name]['std_acc'] = std_acc

        print(f"\n>>> {model_type} - {exp_name}: {mean_acc:.2f}% ± {std_acc:.2f}%\n")

print("\n\n" + "="*70)
print("ALL BASE EXPERIMENTS COMPLETED!")
print("="*70)

## Identify Best Configurations

In [ ]:
# Find best experiment for each model
best_configs = {}

for model_name in config.models:
    model_type = 'Swin' if 'swin' in model_name else 'ViT'

    # Check if experiments completed
    if not all('mean_acc' in all_results[model_name][exp] for exp in experiments):
        print(f" {model_type} experiments not completed yet. Skipping...")
        continue

    best_exp = max(experiments, key=lambda exp: all_results[model_name][exp]['mean_acc'])
    best_acc = all_results[model_name][best_exp]['mean_acc']

    best_configs[model_name] = {
        'experiment': best_exp,
        'mean_acc': best_acc,
        'models': all_results[model_name][best_exp]['folds']
    }

    print(f"{model_type} Best: {best_exp} ({best_acc:.2f}%)")

if best_configs:
    print("\nBest configurations identified")
else:
    print("\n No experiments completed yet. Run the training loop first.")

## Apply TTA to Best Configs

In [ ]:
# We'll evaluate TTA on validation set for best models
print("\n" + "="*70)
print("EVALUATING TTA ON BEST CONFIGURATIONS")
print("="*70)

for model_name in config.models:
    model_type = 'Swin' if 'swin' in model_name else 'ViT'
    best_exp = best_configs[model_name]['experiment']

    print(f"\n{model_type} - {best_exp} + TTA")

    tta_accs = []

    # Evaluate each fold with TTA
    for fold_idx, (train_idx, val_idx) in enumerate(skf.split(combined_df, combined_df['label'])):
        val_fold_df = combined_df.iloc[val_idx]
        model = best_configs[model_name]['models'][fold_idx]['model']

        correct = 0
        total = len(val_fold_df)

        for _, row in tqdm(val_fold_df.iterrows(), total=total, leave=False):
            pred_probs = predict_with_tta(model, row['full_path'], tta_transforms, device)
            pred_class = pred_probs.argmax().item()
            if pred_class == row['label']:
                correct += 1

        fold_acc = 100. * correct / total
        tta_accs.append(fold_acc)
        print(f"  Fold {fold_idx}: {fold_acc:.2f}%")

    tta_mean = np.mean(tta_accs)
    tta_std = np.std(tta_accs)

    # Store TTA results
    all_results[model_name]['best_with_tta'] = {
        'base_experiment': best_exp,
        'mean_acc': tta_mean,
        'std_acc': tta_std
    }

    print(f"  TTA Mean: {tta_mean:.2f}% ± {tta_std:.2f}%")
    print(f"  Improvement: +{tta_mean - best_configs[model_name]['mean_acc']:.2f}%")

## Ensemble Best Models

In [ ]:
print("\n" + "="*70)
print("EVALUATING ENSEMBLE (BEST SWIN + BEST VIT)")
print("="*70)

ensemble_accs = []

for fold_idx, (train_idx, val_idx) in enumerate(skf.split(combined_df, combined_df['label'])):
    val_fold_df = combined_df.iloc[val_idx]

    # Get best model for each architecture from this fold
    swin_model = best_configs[config.models[0]]['models'][fold_idx]['model']
    vit_model = best_configs[config.models[1]]['models'][fold_idx]['model']

    # Weights based on validation accuracy
    swin_acc = best_configs[config.models[0]]['models'][fold_idx]['best_val_acc']
    vit_acc = best_configs[config.models[1]]['models'][fold_idx]['best_val_acc']
    total_acc = swin_acc + vit_acc
    weights = [swin_acc / total_acc, vit_acc / total_acc]

    correct = 0
    total = len(val_fold_df)

    for _, row in tqdm(val_fold_df.iterrows(), total=total, leave=False):
        pred_probs = ensemble_predict([swin_model, vit_model], row['full_path'], val_transform, device, weights)
        pred_class = pred_probs.argmax().item()
        if pred_class == row['label']:
            correct += 1

    fold_acc = 100. * correct / total
    ensemble_accs.append(fold_acc)
    print(f"Fold {fold_idx}: {fold_acc:.2f}%")

ensemble_mean = np.mean(ensemble_accs)
ensemble_std = np.std(ensemble_accs)

all_results['ensemble'] = {
    'mean_acc': ensemble_mean,
    'std_acc': ensemble_std
}

print(f"\nEnsemble Mean: {ensemble_mean:.2f}% ± {ensemble_std:.2f}%")

## Results Summary

In [ ]:
# Create comprehensive results table
results_data = []

for model_name in config.models:
    model_type = 'Swin' if 'swin' in model_name else 'ViT'

    for exp_name in experiments:
        results_data.append({
            'Model': model_type,
            'Experiment': exp_name.replace('_', ' ').title(),
            'Mean Acc': all_results[model_name][exp_name]['mean_acc'],
            'Std': all_results[model_name][exp_name]['std_acc']
        })

    # Add TTA result
    if 'best_with_tta' in all_results[model_name]:
        results_data.append({
            'Model': model_type,
            'Experiment': f"{all_results[model_name]['best_with_tta']['base_experiment'].replace('_', ' ').title()} + TTA",
            'Mean Acc': all_results[model_name]['best_with_tta']['mean_acc'],
            'Std': all_results[model_name]['best_with_tta']['std_acc']
        })

# Add ensemble
results_data.append({
    'Model': 'Both',
    'Experiment': 'Ensemble',
    'Mean Acc': all_results['ensemble']['mean_acc'],
    'Std': all_results['ensemble']['std_acc']
})

results_df = pd.DataFrame(results_data)
results_df['Accuracy'] = results_df.apply(lambda x: f"{x['Mean Acc']:.2f}% ± {x['Std']:.2f}%", axis=1)

print("\n" + "="*70)
print("FINAL RESULTS SUMMARY")
print("="*70)
print(results_df[['Model', 'Experiment', 'Accuracy']].to_string(index=False))

# Visualization
plt.figure(figsize=(12, 6))
results_df['exp_label'] = results_df['Model'] + ' - ' + results_df['Experiment']
plt.barh(results_df['exp_label'], results_df['Mean Acc'])
plt.xlabel('Validation Accuracy (%)')
plt.title('Experiment Results Comparison')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'experiment_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# Save results
results_df.to_csv(OUTPUT_DIR / 'experiment_results.csv', index=False)
print(f"\nResults saved to {OUTPUT_DIR}")

## Save Best Models

In [ ]:
# Save best models for test predictions
save_dir = OUTPUT_DIR / 'best_models'
save_dir.mkdir(exist_ok=True)

for model_name in config.models:
    model_type = 'swin' if 'swin' in model_name else 'vit'
    best_exp = best_configs[model_name]['experiment']

    model_save_dir = save_dir / f"{model_type}_{best_exp}"
    model_save_dir.mkdir(exist_ok=True)

    for fold_idx, fold_result in enumerate(best_configs[model_name]['models']):
        save_path = model_save_dir / f"fold_{fold_idx}.pth"
        torch.save(fold_result['model'].state_dict(), save_path)

    print(f"Saved {model_type} models to {model_save_dir}")

# Save metadata
metadata = {
    'timestamp': datetime.now().isoformat(),
    'best_configs': {k: {'experiment': v['experiment'], 'mean_acc': v['mean_acc']} for k, v in best_configs.items()},
    'ensemble_acc': all_results['ensemble']['mean_acc'],
    'class_mapping': class_mapping
}

with open(OUTPUT_DIR / 'experiment_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print("\nMetadata saved")

## Generate Test Predictions

Use the best performing approach (likely ensemble + TTA)

In [ ]:
# Load test images
test_images = sorted(list(TEST_IMG_DIR.glob('*.jpg')))
print(f"Test images: {len(test_images)}")

# Get best models (all 3 folds for both architectures)
swin_models = [f['model'] for f in best_configs[config.models[0]]['models']]
vit_models = [f['model'] for f in best_configs[config.models[1]]['models']]
all_models = swin_models + vit_models

# Equal weights for ensemble
weights = [1/len(all_models)] * len(all_models)

print("\nGenerating predictions with Ensemble + TTA...")
predictions = []

for img_path in tqdm(test_images):
    # Get ensemble prediction with TTA
    all_tta_preds = []

    for tta_transform in tta_transforms:
        pred = ensemble_predict(all_models, img_path, tta_transform, device, weights)
        all_tta_preds.append(pred)

    # Average TTA predictions
    final_pred = torch.stack(all_tta_preds).mean(0)
    pred_class_idx = final_pred.argmax().item()

    predictions.append(pred_class_idx)

print("Predictions generated")

In [ ]:
# Create submission
idx_to_class = {v: k for k, v in class_mapping.items()}

print(idx_to_class)

submission_df = pd.DataFrame({
    'path': [img.name for img in test_images],
    'class_idx': [pred for pred in predictions]
})

submission_path = OUTPUT_DIR / 'submission.csv'
submission_df.to_csv(submission_path, index=False)

print(f"\n Submission saved to {submission_path}")
print(f"\nFirst few predictions:")
print(submission_df.head(10))

Evaluate performance per class - VIT Progressive unfreezing

In [ ]:
import torch
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
import numpy as np
from pathlib import Path
from PIL import Image
from torchvision import transforms

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model_type = 'vit'
n_folds = 3
num_classes = 20

# Define the same transform used during training/validation
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Create dataset from combined train/val
class EvalDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row['full_path']).convert('RGB')
        label = row['label']

        if self.transform:
            image = self.transform(image)

        return image, label

# Create dataset
eval_dataset = EvalDataset(combined_df, transform=val_transform)
print(f"Evaluation dataset size: {len(eval_dataset)}")

def convert_timm_to_hf_vit(timm_state_dict):
    """Convert timm ViT checkpoint to HuggingFace ViT format."""
    hf_state_dict = {}

    for key, value in timm_state_dict.items():
        # Embeddings
        if key == 'cls_token':
            hf_state_dict['vit.embeddings.cls_token'] = value
        elif key == 'pos_embed':
            hf_state_dict['vit.embeddings.position_embeddings'] = value
        elif key == 'patch_embed.proj.weight':
            hf_state_dict['vit.embeddings.patch_embeddings.projection.weight'] = value
        elif key == 'patch_embed.proj.bias':
            hf_state_dict['vit.embeddings.patch_embeddings.projection.bias'] = value

        # Transformer blocks
        elif key.startswith('blocks.'):
            parts = key.split('.')
            layer_idx = parts[1]

            # QKV weights - split into Q, K, V
            if 'attn.qkv.weight' in key:
                qkv_weight = value
                q, k, v = qkv_weight.chunk(3, dim=0)
                hf_state_dict[f'vit.encoder.layer.{layer_idx}.attention.attention.query.weight'] = q
                hf_state_dict[f'vit.encoder.layer.{layer_idx}.attention.attention.key.weight'] = k
                hf_state_dict[f'vit.encoder.layer.{layer_idx}.attention.attention.value.weight'] = v

            elif 'attn.qkv.bias' in key:
                qkv_bias = value
                q, k, v = qkv_bias.chunk(3, dim=0)
                hf_state_dict[f'vit.encoder.layer.{layer_idx}.attention.attention.query.bias'] = q
                hf_state_dict[f'vit.encoder.layer.{layer_idx}.attention.attention.key.bias'] = k
                hf_state_dict[f'vit.encoder.layer.{layer_idx}.attention.attention.value.bias'] = v

            elif 'attn.proj.weight' in key:
                hf_state_dict[f'vit.encoder.layer.{layer_idx}.attention.output.dense.weight'] = value
            elif 'attn.proj.bias' in key:
                hf_state_dict[f'vit.encoder.layer.{layer_idx}.attention.output.dense.bias'] = value

            elif 'norm1.weight' in key:
                hf_state_dict[f'vit.encoder.layer.{layer_idx}.layernorm_before.weight'] = value
            elif 'norm1.bias' in key:
                hf_state_dict[f'vit.encoder.layer.{layer_idx}.layernorm_before.bias'] = value

            elif 'norm2.weight' in key:
                hf_state_dict[f'vit.encoder.layer.{layer_idx}.layernorm_after.weight'] = value
            elif 'norm2.bias' in key:
                hf_state_dict[f'vit.encoder.layer.{layer_idx}.layernorm_after.bias'] = value

            elif 'mlp.fc1.weight' in key:
                hf_state_dict[f'vit.encoder.layer.{layer_idx}.intermediate.dense.weight'] = value
            elif 'mlp.fc1.bias' in key:
                hf_state_dict[f'vit.encoder.layer.{layer_idx}.intermediate.dense.bias'] = value
            elif 'mlp.fc2.weight' in key:
                hf_state_dict[f'vit.encoder.layer.{layer_idx}.output.dense.weight'] = value
            elif 'mlp.fc2.bias' in key:
                hf_state_dict[f'vit.encoder.layer.{layer_idx}.output.dense.bias'] = value

        # Final LayerNorm
        elif key == 'norm.weight':
            hf_state_dict['vit.layernorm.weight'] = value
        elif key == 'norm.bias':
            hf_state_dict['vit.layernorm.bias'] = value

        # Classifier head
        elif key == 'head.1.weight':
            hf_state_dict['classifier.weight'] = value
        elif key == 'head.1.bias':
            hf_state_dict['classifier.bias'] = value
        elif key == 'head.weight':
            hf_state_dict['classifier.weight'] = value
        elif key == 'head.bias':
            hf_state_dict['classifier.bias'] = value

    return hf_state_dict

# Storage for predictions
all_fold_preds = []
all_fold_labels = []

for fold in range(n_folds):
    print(f"\n{'='*60}")
    print(f"FOLD {fold + 1}/{n_folds}")
    print(f"{'='*60}")

    model_path = OUTPUT_DIR / 'best_models' / 'vit_progressive_unfreeze' / f'fold_{fold}.pth'

    # Load and convert checkpoint
    timm_checkpoint = torch.load(model_path, map_location=device)
    hf_checkpoint = convert_timm_to_hf_vit(timm_checkpoint)

    # Initialize HuggingFace model
    from transformers import ViTForImageClassification
    model = ViTForImageClassification.from_pretrained(
        'google/vit-base-patch16-224',
        num_labels=num_classes,
        ignore_mismatched_sizes=True
    )

    # Load converted weights
    missing, unexpected = model.load_state_dict(hf_checkpoint, strict=False)

    if len(missing) == 0:
        print(f" All weights loaded successfully!")
    else:
        print(f"  Missing keys: {len(missing)}: {missing[:3]}")

    model = model.to(device)
    model.eval()

    # Create dataloader
    eval_loader = DataLoader(
        eval_dataset,
        batch_size=32,
        shuffle=False,
        num_workers=4,
        pin_memory=True
    )

    # Run predictions
    fold_preds = []
    fold_labels = []

    with torch.no_grad():
        for images, labels in tqdm(eval_loader, desc=f'Fold {fold+1} Evaluation'):
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            logits = outputs.logits
            predictions = torch.argmax(logits, dim=1)

            fold_preds.extend(predictions.cpu().numpy())
            fold_labels.extend(labels.cpu().numpy())

    fold_preds = np.array(fold_preds)
    fold_labels = np.array(fold_labels)

    all_fold_preds.append(fold_preds)
    all_fold_labels.append(fold_labels)

    accuracy = (fold_preds == fold_labels).mean()
    print(f" Fold {fold + 1} Accuracy: {accuracy:.4f}")

# Ensemble predictions
all_fold_preds = np.array(all_fold_preds)
all_labels = all_fold_labels[0]

from scipy import stats
ensemble_preds, _ = stats.mode(all_fold_preds, axis=0, keepdims=False)

print(f"\n{'='*60}")
print(f"ENSEMBLE RESULTS (Majority Voting)")
print(f"{'='*60}")
print(f"Total samples: {len(ensemble_preds)}")
print(f" Ensemble Accuracy: {(ensemble_preds == all_labels).mean():.4f}")

print(f"\n{'='*60}")
print(f"INDIVIDUAL FOLD ACCURACIES")
print(f"{'='*60}")
for i, (preds, labels) in enumerate(zip(all_fold_preds, all_fold_labels)):
    acc = (preds == labels).mean()
    print(f"Fold {i+1}: {acc:.4f}")

from sklearn.metrics import classification_report, confusion_matrix

print(f"\n{'='*60}")
print(f"DETAILED METRICS")
print(f"{'='*60}")
print("\nClassification Report:")
print(classification_report(all_labels, ensemble_preds, digits=4, zero_division=0))

print("\nConfusion Matrix:")
cm = confusion_matrix(all_labels, ensemble_preds)
print(cm)

## Detailed Per-Class Evaluation

Evaluate best models (ViT Progressive, Swin Full, Ensemble) with detailed per-class metrics

### ViT Progressive Unfreezing + TTA - Per-Class Metrics

In [ ]:
# Evaluate ViT Progressive (best config) on validation set
print("Evaluating ViT Progressive Unfreezing + TTA")
print("="*80)

# Get class names
class_names = sorted([name for name, idx in class_to_idx.items()], key=lambda x: class_to_idx[x])

# Evaluate each fold
vit_fold_metrics = []
vit_per_class_accs = {class_name: [] for class_name in class_names}

for fold in range(3):
    print(f"\nFold {fold+1}/3")
    print("-"*80)
    
    # Load model
    model_path = OUTPUT_DIR / 'best_models' / 'vit_progressive_unfreeze' / f'fold_{fold}.pth'
    model = create_model('vit', num_classes=20)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model = model.to(device)
    
    # Evaluate
    metrics = evaluate_per_class_metrics(model, combined_loader, class_names, device, f"ViT Fold {fold+1}")
    vit_fold_metrics.append(metrics)
    
    # Store per-class accuracies
    for class_name, acc in metrics['per_class_accuracy'].items():
        vit_per_class_accs[class_name].append(acc)
    
    print(f"Fold {fold+1} Accuracy: {metrics['accuracy']*100:.2f}%")

# Calculate average per-class metrics
print("\n" + "="*80)
print("ViT PROGRESSIVE - AVERAGE PER-CLASS ACCURACY ACROSS FOLDS")
print("="*80)
print(f"{'Class':<30} {'Mean Acc':<15} {'Std':<10}")
print("-"*80)

vit_per_class_summary = {}
for class_name in sorted(class_names):
    accs = vit_per_class_accs[class_name]
    mean_acc = np.mean(accs)
    std_acc = np.std(accs)
    vit_per_class_summary[class_name] = {'mean': mean_acc, 'std': std_acc}
    print(f"{class_name:<30} {mean_acc:>7.1f}%        {std_acc:>5.1f}%")

# Overall stats
overall_accs = [m['accuracy']*100 for m in vit_fold_metrics]
print(f"\nOverall Accuracy: {np.mean(overall_accs):.2f}% ± {np.std(overall_accs):.2f}%")


### Swin Full Fine-tuning + TTA - Per-Class Metrics

In [ ]:
# Evaluate Swin Full Fine-tuning (best config) on validation set
print("Evaluating Swin Full Fine-tuning + TTA")
print("="*80)

# Evaluate each fold
swin_fold_metrics = []
swin_per_class_accs = {class_name: [] for class_name in class_names}

for fold in range(3):
    print(f"\nFold {fold+1}/3")
    print("-"*80)
    
    # Load model
    model_path = OUTPUT_DIR / 'best_models' / 'swin_full_finetune' / f'fold_{fold}.pth'
    model = create_model('swin', num_classes=20)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model = model.to(device)
    
    # Evaluate
    metrics = evaluate_per_class_metrics(model, combined_loader, class_names, device, f"Swin Fold {fold+1}")
    swin_fold_metrics.append(metrics)
    
    # Store per-class accuracies
    for class_name, acc in metrics['per_class_accuracy'].items():
        swin_per_class_accs[class_name].append(acc)
    
    print(f"Fold {fold+1} Accuracy: {metrics['accuracy']*100:.2f}%")

# Calculate average per-class metrics
print("\n" + "="*80)
print("SWIN FULL FINE-TUNING - AVERAGE PER-CLASS ACCURACY ACROSS FOLDS")
print("="*80)
print(f"{'Class':<30} {'Mean Acc':<15} {'Std':<10}")
print("-"*80)

swin_per_class_summary = {}
for class_name in sorted(class_names):
    accs = swin_per_class_accs[class_name]
    mean_acc = np.mean(accs)
    std_acc = np.std(accs)
    swin_per_class_summary[class_name] = {'mean': mean_acc, 'std': std_acc}
    print(f"{class_name:<30} {mean_acc:>7.1f}%        {std_acc:>5.1f}%")

# Overall stats
overall_accs = [m['accuracy']*100 for m in swin_fold_metrics]
print(f"\nOverall Accuracy: {np.mean(overall_accs):.2f}% ± {np.std(overall_accs):.2f}%")


### Ensemble (ViT + Swin) - Per-Class Metrics

In [ ]:
# Evaluate Ensemble on validation set
print("Evaluating ViT + Swin Ensemble")
print("="*80)

# Get ensemble predictions by combining ViT and Swin predictions
all_labels = []
all_ensemble_preds = []

# Use weighted averaging of probabilities from all folds
for batch_idx, (images, labels) in enumerate(tqdm(combined_loader, desc="Ensemble Evaluation")):
    images = images.to(device)
    labels = labels.to(device)
    
    batch_probs = []
    
    # Get predictions from all ViT folds
    for fold in range(3):
        model_path = OUTPUT_DIR / 'best_models' / 'vit_progressive_unfreeze' / f'fold_{fold}.pth'
        model = create_model('vit', num_classes=20)
        model.load_state_dict(torch.load(model_path, map_location=device))
        model = model.to(device)
        model.eval()
        
        with torch.no_grad():
            outputs = model(images)
            if hasattr(outputs, 'logits'):
                outputs = outputs.logits
            probs = torch.nn.functional.softmax(outputs, dim=1)
            batch_probs.append(probs.cpu().numpy())
    
    # Get predictions from all Swin folds
    for fold in range(3):
        model_path = OUTPUT_DIR / 'best_models' / 'swin_full_finetune' / f'fold_{fold}.pth'
        model = create_model('swin', num_classes=20)
        model.load_state_dict(torch.load(model_path, map_location=device))
        model = model.to(device)
        model.eval()
        
        with torch.no_grad():
            outputs = model(images)
            if hasattr(outputs, 'logits'):
                outputs = outputs.logits
            probs = torch.nn.functional.softmax(outputs, dim=1)
            batch_probs.append(probs.cpu().numpy())
    
    # Average probabilities
    avg_probs = np.mean(batch_probs, axis=0)
    preds = np.argmax(avg_probs, axis=1)
    
    all_ensemble_preds.extend(preds)
    all_labels.extend(labels.cpu().numpy())

# Calculate metrics
accuracy = np.mean(np.array(all_ensemble_preds) == np.array(all_labels))

# Per-class accuracies
ensemble_per_class_acc = {}
for i, class_name in enumerate(class_names):
    class_mask = np.array(all_labels) == i
    if class_mask.sum() > 0:
        class_correct = np.sum((np.array(all_ensemble_preds)[class_mask] == i))
        ensemble_per_class_acc[class_name] = class_correct / class_mask.sum() * 100
    else:
        ensemble_per_class_acc[class_name] = 0.0

print("\n" + "="*80)
print("ENSEMBLE (ViT + Swin) - PER-CLASS ACCURACY")
print("="*80)
print(f"\nOverall Accuracy: {accuracy*100:.2f}%\n")
print(f"{'Class':<30} {'Accuracy':<12}")
print("-"*80)

for class_name in sorted(class_names):
    acc = ensemble_per_class_acc[class_name]
    print(f"{class_name:<30} {acc:>7.1f}%")


### Save Per-Class Metrics to CSV

In [ ]:
# Compile all per-class metrics into a single CSV
print("Compiling per-class metrics...")

rows = []
for class_name in sorted(class_names):
    row = {'Class': class_name}
    
    # ViT metrics
    row['ViT_Mean'] = f"{vit_per_class_summary[class_name]['mean']:.1f}"
    row['ViT_Std'] = f"{vit_per_class_summary[class_name]['std']:.1f}"
    
    # Swin metrics
    row['Swin_Mean'] = f"{swin_per_class_summary[class_name]['mean']:.1f}"
    row['Swin_Std'] = f"{swin_per_class_summary[class_name]['std']:.1f}"
    
    # Ensemble metrics
    row['Ensemble'] = f"{ensemble_per_class_acc[class_name]:.1f}"
    
    rows.append(row)

df_per_class = pd.DataFrame(rows)

# Save to CSV
output_path = OUTPUT_DIR / 'transformer_per_class_metrics.csv'
df_per_class.to_csv(output_path, index=False)
print(f"\nPer-class metrics saved to: {output_path}")
print("\n" + df_per_class.to_string(index=False))

# Also save summary metrics
summary_rows = [
    {
        'Model': 'ViT Progressive + TTA',
        'Val_Accuracy': f"{np.mean([m['accuracy']*100 for m in vit_fold_metrics]):.2f}",
        'Val_Std': f"{np.std([m['accuracy']*100 for m in vit_fold_metrics]):.2f}"
    },
    {
        'Model': 'Swin Full Finetune + TTA',
        'Val_Accuracy': f"{np.mean([m['accuracy']*100 for m in swin_fold_metrics]):.2f}",
        'Val_Std': f"{np.std([m['accuracy']*100 for m in swin_fold_metrics]):.2f}"
    },
    {
        'Model': 'Ensemble (ViT + Swin)',
        'Val_Accuracy': f"{accuracy*100:.2f}",
        'Val_Std': '0.00'
    }
]

df_summary = pd.DataFrame(summary_rows)
summary_path = OUTPUT_DIR / 'transformer_summary_metrics.csv'
df_summary.to_csv(summary_path, index=False)
print(f"\nSummary metrics saved to: {summary_path}")
print("\n" + df_summary.to_string(index=False))
